# 06 · GroupBy and Aggregation

**Goal:** master `groupby()` — pandas' equivalent of SQL's `GROUP BY` — for summarizing data
by category, plus multi-level grouping and pivot-style reshaping.

### Setup

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "department": ["Sales", "Engineering", "Sales", "Marketing", "Engineering", "Sales", "Marketing"],
    "employee": ["Alice", "Bob", "Charlie", "Diana", "Evan", "Fiona", "George"],
    "salary": [55000, 85000, 48000, 62000, 91000, 51000, 58000],
    "years_experience": [2, 5, 1, 3, 7, 2, 4]
})
print(df)

    department employee  salary  years_experience
0        Sales    Alice   55000                 2
1  Engineering      Bob   85000                 5
2        Sales  Charlie   48000                 1
3    Marketing    Diana   62000                 3
4  Engineering     Evan   91000                 7
5        Sales    Fiona   51000                 2
6    Marketing   George   58000                 4


### The "split-apply-combine" mental model

`groupby()` works in three conceptual steps:

1. **Split** the data into groups based on a column's values
2. **Apply** an aggregation function (sum, mean, count...) to each group independently
3. **Combine** the results back into a single Series/DataFrame

```
department  salary                    department  mean_salary
Sales       55000    ---split--->      Sales:    [55000,48000,51000] --apply(mean)--> 51333
Eng.        85000                      Eng.:     [85000,91000]       --apply(mean)--> 88000
Sales       48000                      Marketing:[62000,58000]        --apply(mean)--> 60000
Marketing   62000
Eng.        91000
Sales       51000
Marketing   58000
```

In [2]:
grouped = df.groupby("department")
print(type(grouped))    # a special DataFrameGroupBy object -- nothing computed yet!

# Aggregating: this is where the actual computation happens
print(grouped["salary"].mean())    # average salary PER department

<class 'pandas.api.typing.DataFrameGroupBy'>
department
Engineering    88000.000000
Marketing      60000.000000
Sales          51333.333333
Name: salary, dtype: float64


### Common aggregation functions

In [3]:
print(df.groupby("department")["salary"].sum())      # total salary per department
print()
print(df.groupby("department")["salary"].count())     # number of employees per department
print()
print(df.groupby("department")["salary"].min())        # lowest salary per department
print()
print(df.groupby("department")["salary"].max())        # highest salary per department
print()
print(df.groupby("department").size())                  # row count per group (all columns)

department
Engineering    176000
Marketing      120000
Sales          154000
Name: salary, dtype: int64

department
Engineering    2
Marketing      2
Sales          3
Name: salary, dtype: int64

department
Engineering    85000
Marketing      58000
Sales          48000
Name: salary, dtype: int64

department
Engineering    91000
Marketing      62000
Sales          55000
Name: salary, dtype: int64

department
Engineering    2
Marketing      2
Sales          3
dtype: int64


### Aggregating multiple columns / multiple functions at once with `.agg()`

In [4]:
# One function, multiple columns
print(df.groupby("department")[["salary", "years_experience"]].mean())

                   salary  years_experience
department                                 
Engineering  88000.000000          6.000000
Marketing    60000.000000          3.500000
Sales        51333.333333          1.666667


In [5]:
# Multiple functions on one column
print(df.groupby("department")["salary"].agg(["mean", "min", "max", "count"]))

                     mean    min    max  count
department                                    
Engineering  88000.000000  85000  91000      2
Marketing    60000.000000  58000  62000      2
Sales        51333.333333  48000  55000      3


In [6]:
# Different functions for different columns -- using a dict
summary = df.groupby("department").agg({
    "salary": ["mean", "max"],
    "years_experience": "sum",
    "employee": "count"
})
print(summary)

                   salary        years_experience employee
                     mean    max              sum    count
department                                                
Engineering  88000.000000  91000               12        2
Marketing    60000.000000  62000                7        2
Sales        51333.333333  55000                5        3


### Named aggregation — cleaner column names for the result

In [7]:
summary = df.groupby("department").agg(
    avg_salary=("salary", "mean"),
    max_salary=("salary", "max"),
    total_experience=("years_experience", "sum"),
    num_employees=("employee", "count")
)
print(summary)

               avg_salary  max_salary  total_experience  num_employees
department                                                            
Engineering  88000.000000       91000                12              2
Marketing    60000.000000       62000                 7              2
Sales        51333.333333       55000                 5              3


### Grouping by MULTIPLE columns

Produces a hierarchical (multi-level) index in the result — one level per grouping column.

In [8]:
df2 = df.copy()
df2["seniority"] = np.where(df2["years_experience"] >= 4, "Senior", "Junior")

grouped2 = df2.groupby(["department", "seniority"])["salary"].mean()
print(grouped2)
print()
print(type(grouped2.index))    # a MultiIndex

# .reset_index() turns the group labels back into normal columns -- often more convenient
print(grouped2.reset_index())

department   seniority
Engineering  Senior       88000.000000
Marketing    Junior       62000.000000
             Senior       58000.000000
Sales        Junior       51333.333333
Name: salary, dtype: float64

<class 'pandas.MultiIndex'>
    department seniority        salary
0  Engineering    Senior  88000.000000
1    Marketing    Junior  62000.000000
2    Marketing    Senior  58000.000000
3        Sales    Junior  51333.333333


### Iterating over groups (rarely needed, but useful to understand what's happening)

In [9]:
for department, group_df in df.groupby("department"):
    print(f"--- {department} ---")
    print(group_df[["employee", "salary"]])
    print()

--- Engineering ---
  employee  salary
1      Bob   85000
4     Evan   91000

--- Marketing ---
  employee  salary
3    Diana   62000
6   George   58000

--- Sales ---
  employee  salary
0    Alice   55000
2  Charlie   48000
5    Fiona   51000



### `.transform()` — aggregate but keep the ORIGINAL shape

Unlike `.agg()` (which collapses each group to one row), `.transform()` broadcasts the group's
result back to every original row — perfect for things like "difference from group average".

In [10]:
df["dept_avg_salary"] = df.groupby("department")["salary"].transform("mean")
df["salary_vs_dept_avg"] = df["salary"] - df["dept_avg_salary"]

print(df[["employee", "department", "salary", "dept_avg_salary", "salary_vs_dept_avg"]])

  employee   department  salary  dept_avg_salary  salary_vs_dept_avg
0    Alice        Sales   55000     51333.333333         3666.666667
1      Bob  Engineering   85000     88000.000000        -3000.000000
2  Charlie        Sales   48000     51333.333333        -3333.333333
3    Diana    Marketing   62000     60000.000000         2000.000000
4     Evan  Engineering   91000     88000.000000         3000.000000
5    Fiona        Sales   51000     51333.333333         -333.333333
6   George    Marketing   58000     60000.000000        -2000.000000


### `pivot_table` — a groupby/reshape combo, very Excel-like

`pivot_table` groups by one column for rows, another for columns, and aggregates a third —
directly mirroring an Excel pivot table.

In [11]:
df3 = pd.DataFrame({
    "department": ["Sales", "Sales", "Engineering", "Engineering", "Marketing", "Marketing"],
    "quarter": ["Q1", "Q2", "Q1", "Q2", "Q1", "Q2"],
    "revenue": [10000, 12000, 25000, 27000, 8000, 9500]
})

pivot = df3.pivot_table(values="revenue", index="department", columns="quarter", aggfunc="sum")
print(pivot)

quarter         Q1     Q2
department               
Engineering  25000  27000
Marketing     8000   9500
Sales        10000  12000


### 🧠 Quick check

1. What are the three conceptual steps behind `groupby()`?
2. What's the difference between `.agg("mean")` and `.transform("mean")` after a `groupby()`?
3. What does grouping by two columns produce as the resulting index?

<details>
<summary>Answers</summary>

1. Split (divide data into groups), apply (run an aggregation function per group), combine
   (assemble results back into one Series/DataFrame).
2. `.agg()` collapses each group down to one summary row; `.transform()` keeps the original
   number of rows, broadcasting each group's result back to every row in that group.
3. A hierarchical `MultiIndex`, with one level per grouping column (use `.reset_index()` to
   flatten it back into regular columns).
</details>

### ✍️ Practice

1. Group the sample DataFrame by `department` and compute the average `years_experience`.
2. Use named aggregation to compute, per department, the minimum salary, maximum salary, and
   number of employees in one call.
3. Group by both `department` and a new `seniority` column, and find the average salary for
   each combination.
4. Use `.transform()` to add a column showing each employee's salary as a percentage of their
   department's total salary.

Continue to **`07_merging_joining_concatenating.ipynb`** next.